In [ ]:
# ============================================================
# FINAL THESIS ROBUSTNESS ANALYSES
# Cell 1 — Mount Google Drive and define project paths
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

BASE = Path("/content/drive/MyDrive/Master/Thesis/Code")

PH4 = BASE / "processed_data" / "PH4"

NAB_V2 = (
    PH4
    / "supervised"
    / "NAB"
    / "feature_set_v2"
)

NAB_METRICS = (
    NAB_V2
    / "metrics"
    / "nab_supervised_feature_set_v2_metrics_deduplicated.csv"
)

NAB_PREDICTIONS = (
    NAB_V2
    / "predictions"
    / "nab_supervised_feature_set_v2_predictions_deduplicated.csv"
)

OUT = (
    BASE
    / "experiments"
    / "EXP_FINAL_SUPERVISOR_ROBUSTNESS"
    / "NAB_SUPERVISED_UNCERTAINTY"
)

OUT.mkdir(
    parents=True,
    exist_ok=True
)

print("=" * 90)
print("FINAL THESIS ROBUSTNESS ANALYSES")
print("=" * 90)

print("\nBASE:")
print(BASE)

print("\nNAB metrics:")
print(NAB_METRICS)
print("Exists:", NAB_METRICS.exists())

print("\nNAB predictions:")
print(NAB_PREDICTIONS)
print("Exists:", NAB_PREDICTIONS.exists())

print("\nOutput directory:")
print(OUT)
print("Exists:", OUT.exists())

assert BASE.exists(), "Thesis BASE directory not found."
assert NAB_METRICS.exists(), "Final NAB V2 metrics file not found."
assert NAB_PREDICTIONS.exists(), "Final NAB V2 predictions file not found."

print("\nPASS: required final NAB artifacts are available.")

In [ ]:
# ============================================================
# FINAL THESIS ROBUSTNESS ANALYSES
# Cell 2 — Audit final NAB supervised test predictions
# ============================================================

import pandas as pd
import numpy as np

from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
)

# ------------------------------------------------------------
# 1. Load authoritative saved artifacts
# ------------------------------------------------------------
pred = pd.read_csv(NAB_PREDICTIONS)
stored_metrics = pd.read_csv(NAB_METRICS)

print("=" * 100)
print("NAB SUPERVISED FEATURE SET V2 — FINAL TEST PREDICTION AUDIT")
print("=" * 100)

print("\nPredictions shape:", pred.shape)
print("Metrics shape:    ", stored_metrics.shape)

print("\nPrediction columns:")
for c in pred.columns:
    print(" -", c)

# ------------------------------------------------------------
# 2. Required-column audit
# ------------------------------------------------------------
required_cols = [
    "model",
    "y_true",
    "y_score",
    "y_pred_validation_tuned",
    "validation_selected_threshold",
]

missing = [c for c in required_cols if c not in pred.columns]

assert not missing, f"Missing required prediction columns: {missing}"

assert pred["y_true"].notna().all(), "Missing y_true values."
assert pred["y_score"].notna().all(), "Missing y_score values."
assert pred["y_pred_validation_tuned"].notna().all(), \
    "Missing validation-tuned predictions."

assert set(pred["y_true"].unique()).issubset({0, 1}), \
    f"Unexpected y_true values: {pred['y_true'].unique()}"

assert set(pred["y_pred_validation_tuned"].unique()).issubset({0, 1}), \
    "Validation-tuned predictions are not binary."

# ------------------------------------------------------------
# 3. Recompute final-test metrics independently
# ------------------------------------------------------------
audit_rows = []

for model, dfm in pred.groupby("model", sort=True):

    dfm = dfm.reset_index(drop=True)

    y_true = dfm["y_true"].astype(int).to_numpy()
    y_pred = dfm["y_pred_validation_tuned"].astype(int).to_numpy()
    y_score = dfm["y_score"].astype(float).to_numpy()

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    thresholds = (
        dfm["validation_selected_threshold"]
        .dropna()
        .astype(float)
        .unique()
    )

    assert len(thresholds) == 1, (
        f"{model}: expected exactly one validation-selected "
        f"threshold, found {thresholds}"
    )

    threshold = float(thresholds[0])

    precision = precision_score(
        y_true, y_pred, zero_division=0
    )
    recall = recall_score(
        y_true, y_pred, zero_division=0
    )
    f1 = f1_score(
        y_true, y_pred, zero_division=0
    )
    roc_auc = roc_auc_score(
        y_true, y_score
    )
    ap = average_precision_score(
        y_true, y_score
    )

    audit_rows.append({
        "model": model,
        "n_test": len(dfm),
        "positive_test": int(y_true.sum()),
        "negative_test": int((y_true == 0).sum()),
        "predicted_positive": int(y_pred.sum()),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp),
        "threshold": threshold,
        "precision_recomputed": precision,
        "recall_recomputed": recall,
        "f1_recomputed": f1,
        "roc_auc_recomputed": roc_auc,
        "average_precision_recomputed": ap,
    })

audit = pd.DataFrame(audit_rows)

# ------------------------------------------------------------
# 4. Verify common final-test population
# ------------------------------------------------------------
assert (audit["n_test"] == 13548).all(), (
    "Unexpected test-set size for at least one model."
)

assert (audit["positive_test"] == 48).all(), (
    "Expected exactly 48 positive test windows for every model."
)

assert (audit["negative_test"] == 13500).all(), (
    "Expected exactly 13,500 negative test windows for every model."
)

# Check that y_true is identical across models in stored row order
model_groups = {
    model: dfm.reset_index(drop=True)["y_true"].astype(int).to_numpy()
    for model, dfm in pred.groupby("model", sort=True)
}

reference_model = list(model_groups.keys())[0]
reference_y = model_groups[reference_model]

for model, y in model_groups.items():
    assert np.array_equal(reference_y, y), (
        f"y_true row sequence differs between "
        f"{reference_model} and {model}."
    )

# ------------------------------------------------------------
# 5. Save provenance audit
# ------------------------------------------------------------
audit_path = OUT / "nab_supervised_final_test_prediction_audit.csv"
audit.to_csv(audit_path, index=False)

# ------------------------------------------------------------
# 6. Display
# ------------------------------------------------------------
print("\nPASS: all models use the same final test population.")
print("PASS: n_test = 13,548 for every model.")
print("PASS: positives = 48 and negatives = 13,500.")
print("PASS: metrics were recomputed from saved predictions only.")
print("PASS: NO model retraining occurred.")

print("\nRecomputed final-test results:")
display(
    audit[
        [
            "model",
            "TN", "FP", "FN", "TP",
            "threshold",
            "precision_recomputed",
            "recall_recomputed",
            "f1_recomputed",
            "roc_auc_recomputed",
            "average_precision_recomputed",
        ]
    ].round(6)
)

print("\nStored metrics file:")
display(stored_metrics)

print("\nAudit saved to:")
print(audit_path)

In [ ]:
# ============================================================
# FINAL THESIS ROBUSTNESS ANALYSES
# Cell 3 — NAB supervised F1 sensitivity to ±1 / ±2
#          positive-window classification outcomes
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# Helper: F1 directly from confusion-matrix counts
# ------------------------------------------------------------
def f1_from_counts(tp, fp, fn):
    denom = 2 * tp + fp + fn
    return 0.0 if denom == 0 else (2 * tp) / denom


# ------------------------------------------------------------
# Construct controlled perturbations
#
# "FN -> TP" means one/two previously missed positive windows
# become correctly detected.
#
# "TP -> FN" means one/two previously detected positive windows
# become missed.
#
# Ground-truth labels and all negative-window outcomes remain fixed.
# ------------------------------------------------------------
sensitivity_rows = []

for _, row in audit.iterrows():

    model = row["model"]

    tp0 = int(row["TP"])
    fp0 = int(row["FP"])
    fn0 = int(row["FN"])

    baseline_f1 = f1_from_counts(tp0, fp0, fn0)

    scenarios = [
        ("-2 detected positives", -2),
        ("-1 detected positive",  -1),
        ("Baseline",               0),
        ("+1 detected positive",   1),
        ("+2 detected positives",  2),
    ]

    for scenario_name, delta_tp in scenarios:

        # Positive outcome moved from TP -> FN when delta_tp < 0
        # or from FN -> TP when delta_tp > 0
        new_tp = tp0 + delta_tp
        new_fn = fn0 - delta_tp

        # A scenario is feasible only when the required TP/FN exists.
        feasible = (
            new_tp >= 0
            and new_fn >= 0
        )

        if feasible:
            new_f1 = f1_from_counts(
                tp=new_tp,
                fp=fp0,
                fn=new_fn
            )

            delta_f1 = new_f1 - baseline_f1

        else:
            new_f1 = np.nan
            delta_f1 = np.nan

        sensitivity_rows.append({
            "model": model,
            "scenario": scenario_name,
            "delta_detected_positive_windows": delta_tp,
            "baseline_TP": tp0,
            "baseline_FP": fp0,
            "baseline_FN": fn0,
            "scenario_TP": new_tp if feasible else np.nan,
            "scenario_FP": fp0 if feasible else np.nan,
            "scenario_FN": new_fn if feasible else np.nan,
            "baseline_f1": baseline_f1,
            "scenario_f1": new_f1,
            "absolute_f1_change": delta_f1,
            "percentage_point_change": (
                delta_f1 * 100
                if feasible else np.nan
            ),
            "feasible": feasible,
        })


sensitivity = pd.DataFrame(sensitivity_rows)

# ------------------------------------------------------------
# Validation checks
# ------------------------------------------------------------
baseline_rows = sensitivity[
    sensitivity["delta_detected_positive_windows"] == 0
].copy()

audit_f1 = (
    audit
    .set_index("model")["f1_recomputed"]
    .sort_index()
)

sens_f1 = (
    baseline_rows
    .set_index("model")["scenario_f1"]
    .sort_index()
)

assert np.allclose(
    audit_f1.values,
    sens_f1.values,
    atol=1e-12
), "Sensitivity baseline does not reproduce audited F1."

# Every feasible scenario must still contain 48 positive windows
feasible_rows = sensitivity[sensitivity["feasible"]].copy()

assert (
    feasible_rows["scenario_TP"]
    + feasible_rows["scenario_FN"]
).eq(48).all(), (
    "Positive test count changed unexpectedly."
)

# ------------------------------------------------------------
# Save full results
# ------------------------------------------------------------
sensitivity_path = (
    OUT
    / "nab_supervised_f1_positive_outcome_sensitivity.csv"
)

sensitivity.to_csv(
    sensitivity_path,
    index=False
)

# ------------------------------------------------------------
# Compact thesis-facing table
# ------------------------------------------------------------
table = (
    sensitivity
    .pivot(
        index="model",
        columns="scenario",
        values="scenario_f1"
    )
    .reset_index()
)

desired_order = [
    "model",
    "-2 detected positives",
    "-1 detected positive",
    "Baseline",
    "+1 detected positive",
    "+2 detected positives",
]

table = table[
    [c for c in desired_order if c in table.columns]
]

table_path = (
    OUT
    / "nab_supervised_f1_positive_outcome_sensitivity_table.csv"
)

table.to_csv(
    table_path,
    index=False
)

# ------------------------------------------------------------
# Display full sensitivity results
# ------------------------------------------------------------
print("=" * 100)
print("NAB SUPERVISED F1 POSITIVE-OUTCOME SENSITIVITY")
print("=" * 100)

print(
    "\nInterpretation: only the classification outcomes of "
    "1 or 2 positive test windows are changed."
)

print(
    "The 48 ground-truth positive windows, all negative-window "
    "outcomes, thresholds, scores, and models remain unchanged."
)

display(
    sensitivity[
        [
            "model",
            "scenario",
            "scenario_TP",
            "scenario_FP",
            "scenario_FN",
            "scenario_f1",
            "percentage_point_change",
            "feasible",
        ]
    ].round(6)
)

print("\nCompact F1 table:")
display(table.round(6))

# ------------------------------------------------------------
# Model-level maximum sensitivity summary
# ------------------------------------------------------------
summary_rows = []

for model, dfm in sensitivity.groupby("model"):

    valid = dfm[dfm["feasible"]].copy()
    baseline = float(
        valid.loc[
            valid["delta_detected_positive_windows"] == 0,
            "scenario_f1"
        ].iloc[0]
    )

    min_f1 = valid["scenario_f1"].min()
    max_f1 = valid["scenario_f1"].max()

    summary_rows.append({
        "model": model,
        "baseline_f1": baseline,
        "minimum_f1_within_plus_minus_2": min_f1,
        "maximum_f1_within_plus_minus_2": max_f1,
        "full_f1_range": max_f1 - min_f1,
        "full_range_percentage_points": (
            max_f1 - min_f1
        ) * 100,
    })

sensitivity_summary = pd.DataFrame(summary_rows)

summary_path = (
    OUT
    / "nab_supervised_f1_positive_outcome_sensitivity_summary.csv"
)

sensitivity_summary.to_csv(
    summary_path,
    index=False
)

print("\nSensitivity summary:")
display(sensitivity_summary.round(6))

# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------
plot_order = [
    "LogisticRegression",
    "MLP",
    "RandomForest",
    "XGBoost",
]

fig, ax = plt.subplots(figsize=(10, 6))

for model in plot_order:

    dfm = sensitivity[
        (sensitivity["model"] == model)
        & (sensitivity["feasible"])
    ].sort_values(
        "delta_detected_positive_windows"
    )

    ax.plot(
        dfm["delta_detected_positive_windows"],
        dfm["scenario_f1"],
        marker="o",
        label=model
    )

ax.set_xlabel(
    "Change in number of correctly detected positive test windows"
)
ax.set_ylabel("F1-Score")
ax.set_title(
    "NAB Supervised Final-Test F1 Sensitivity\n"
    "to ±1 / ±2 Positive-Window Classification Outcomes"
)
ax.set_xticks([-2, -1, 0, 1, 2])
ax.axvline(0, linestyle="--", linewidth=1)
ax.legend()
ax.grid(alpha=0.25)

plt.tight_layout()

plot_path = (
    OUT
    / "nab_supervised_f1_positive_outcome_sensitivity.png"
)

plt.savefig(
    plot_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("\nPASS: sensitivity analysis completed without retraining.")
print("\nSaved full results:")
print(sensitivity_path)

print("\nSaved thesis-facing table:")
print(table_path)

print("\nSaved summary:")
print(summary_path)

print("\nSaved figure:")
print(plot_path)

In [ ]:
# ============================================================
# FINAL THESIS ROBUSTNESS ANALYSES
# Cell 4 — Thesis-ready NAB F1 sensitivity summary
# ============================================================

thesis_rows = []

for model, dfm in sensitivity.groupby("model"):

    baseline_row = dfm[
        dfm["delta_detected_positive_windows"] == 0
    ].iloc[0]

    plus1 = dfm[
        dfm["delta_detected_positive_windows"] == 1
    ].iloc[0]

    minus1_candidates = dfm[
        dfm["delta_detected_positive_windows"] == -1
    ]

    baseline = float(baseline_row["scenario_f1"])

    plus1_f1 = (
        float(plus1["scenario_f1"])
        if plus1["feasible"]
        else np.nan
    )

    if (
        len(minus1_candidates) == 1
        and minus1_candidates.iloc[0]["feasible"]
    ):
        minus1_f1 = float(
            minus1_candidates.iloc[0]["scenario_f1"]
        )
    else:
        minus1_f1 = np.nan

    thesis_rows.append({
        "model": model,
        "baseline_f1": baseline,
        "minus_1_detected_positive_f1": minus1_f1,
        "plus_1_detected_positive_f1": plus1_f1,
        "minus_1_absolute_change": (
            minus1_f1 - baseline
            if not np.isnan(minus1_f1)
            else np.nan
        ),
        "plus_1_absolute_change": (
            plus1_f1 - baseline
            if not np.isnan(plus1_f1)
            else np.nan
        ),
        "plus_1_relative_change_percent": (
            ((plus1_f1 - baseline) / baseline) * 100
            if baseline > 0 and not np.isnan(plus1_f1)
            else np.nan
        ),
    })

thesis_summary = pd.DataFrame(thesis_rows)

thesis_summary_path = (
    OUT
    / "nab_supervised_f1_sensitivity_thesis_summary.csv"
)

thesis_summary.to_csv(
    thesis_summary_path,
    index=False
)

print("=" * 90)
print("THESIS-READY NAB F1 SENSITIVITY SUMMARY")
print("=" * 90)

display(thesis_summary.round(6))

print("\nSaved to:")
print(thesis_summary_path)

print(
    "\nPASS: NAB supervised uncertainty/sensitivity analysis "
    "is complete."
)

In [ ]:
# ============================================================
# FINAL THESIS ROBUSTNESS ANALYSES
# Cell 5 — Locate authoritative residential anomaly-score files
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import os

# ------------------------------------------------------------
# Known final/corrected experiment roots
# ------------------------------------------------------------
UK_EXP = BASE / "experiments" / "EXP_UKDALE_Selected_Pipeline"
AMP_EXP = BASE / "experiments" / "EXP_AMPDS2_Selected_Pipeline"

CROSS_EXP = (
    BASE
    / "experiments"
    / "EXP_UKDALE_AMPDS2_Cross_Model_Comparison"
)

CORRECTION_EXP = (
    BASE
    / "experiments"
    / "EXP_SPLIT_BOUNDARY_PURGE_CORRECTION"
)

roots = {
    "UK_SELECTED": UK_EXP,
    "AMP_SELECTED": AMP_EXP,
    "CROSS_MODEL": CROSS_EXP,
    "BOUNDARY_CORRECTION": CORRECTION_EXP,
}

print("=" * 110)
print("RESIDENTIAL SCORE-ARTIFACT INVENTORY")
print("=" * 110)

for name, root in roots.items():
    print(f"\n{name}:")
    print(root)
    print("Exists:", root.exists())

# ------------------------------------------------------------
# Search likely score/result files
# ------------------------------------------------------------
extensions = {
    ".csv",
    ".parquet",
    ".feather",
    ".npz",
    ".npy",
}

keywords = [
    "score",
    "scores",
    "prediction",
    "predictions",
    "flag",
    "result",
    "window",
    "split",
    "isolation",
    "if_",
]

candidate_files = []

for root_name, root in roots.items():

    if not root.exists():
        continue

    for p in root.rglob("*"):

        if not p.is_file():
            continue

        if p.suffix.lower() not in extensions:
            continue

        name_lower = p.name.lower()

        if any(k in name_lower for k in keywords):

            candidate_files.append(
                {
                    "root": root_name,
                    "path": p,
                    "size_mb": p.stat().st_size / (1024 ** 2),
                }
            )

print(f"\nCandidate files found: {len(candidate_files)}")

# ------------------------------------------------------------
# Inspect schemas without loading full large CSVs
# ------------------------------------------------------------
inventory_rows = []

for item in candidate_files:

    p = item["path"]

    columns_or_keys = ""
    inspection_status = "OK"

    try:

        if p.suffix.lower() == ".csv":

            sample = pd.read_csv(p, nrows=5)
            columns_or_keys = ", ".join(sample.columns.astype(str))

        elif p.suffix.lower() == ".parquet":

            sample = pd.read_parquet(p)
            columns_or_keys = ", ".join(sample.columns.astype(str))

        elif p.suffix.lower() == ".feather":

            sample = pd.read_feather(p)
            columns_or_keys = ", ".join(sample.columns.astype(str))

        elif p.suffix.lower() == ".npz":

            with np.load(p, allow_pickle=True) as z:
                columns_or_keys = ", ".join(z.files)

        elif p.suffix.lower() == ".npy":

            arr = np.load(p, mmap_mode="r", allow_pickle=True)
            columns_or_keys = f"ndarray shape={arr.shape}, dtype={arr.dtype}"

    except Exception as e:

        inspection_status = f"ERROR: {type(e).__name__}: {e}"

    text_lower = columns_or_keys.lower()

    has_score_field = any(
        token in text_lower
        for token in [
            "score",
            "anomaly_score",
            "decision_function",
            "reconstruction_error",
        ]
    )

    has_split_field = any(
        token in text_lower
        for token in [
            "split",
            "partition",
            "train",
            "validation",
            "val",
            "test",
        ]
    )

    inventory_rows.append(
        {
            "root": item["root"],
            "file": p.name,
            "relative_path": str(p.relative_to(BASE)),
            "size_mb": item["size_mb"],
            "has_score_field": has_score_field,
            "has_split_field": has_split_field,
            "columns_or_keys": columns_or_keys,
            "inspection_status": inspection_status,
        }
    )

inventory = pd.DataFrame(inventory_rows)

# Prefer files that visibly contain scores
if len(inventory):

    inventory = inventory.sort_values(
        by=[
            "has_score_field",
            "has_split_field",
            "root",
            "file",
        ],
        ascending=[
            False,
            False,
            True,
            True,
        ],
    ).reset_index(drop=True)

# ------------------------------------------------------------
# Display strongest candidates first
# ------------------------------------------------------------
print("\nFiles containing score-like fields:")
score_candidates = inventory[
    inventory["has_score_field"]
].copy()

if len(score_candidates):

    display(
        score_candidates[
            [
                "root",
                "file",
                "size_mb",
                "has_split_field",
                "relative_path",
                "columns_or_keys",
            ]
        ]
    )

else:

    print("No files with obvious score columns found.")

print("\nAll candidate artifacts:")
display(
    inventory[
        [
            "root",
            "file",
            "size_mb",
            "has_score_field",
            "has_split_field",
            "relative_path",
        ]
    ]
)

# ------------------------------------------------------------
# Save inventory
# ------------------------------------------------------------
THRESHOLD_OUT = (
    BASE
    / "experiments"
    / "EXP_FINAL_SUPERVISOR_ROBUSTNESS"
    / "RESIDENTIAL_THRESHOLD_SENSITIVITY"
)

THRESHOLD_OUT.mkdir(
    parents=True,
    exist_ok=True
)

inventory_path = (
    THRESHOLD_OUT
    / "residential_score_artifact_inventory.csv"
)

inventory.to_csv(
    inventory_path,
    index=False
)

print("\nInventory saved to:")
print(inventory_path)

print(
    "\nNEXT STEP: identify the corrected validation/test "
    "Isolation Forest score artifacts before calculating "
    "90th/95th/99th percentile thresholds."
)

In [ ]:
# ============================================================
# FINAL THESIS ROBUSTNESS ANALYSES
# Cell 6 — Identify authoritative row-level IF score artifacts
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import os

print("=" * 110)
print("AUTHORITATIVE RESIDENTIAL ISOLATION-FOREST SCORE SEARCH")
print("=" * 110)

# ------------------------------------------------------------
# Search roots in priority order
# ------------------------------------------------------------
search_roots = [
    ("BOUNDARY_CORRECTION", CORRECTION_EXP),
    ("UK_SELECTED", UK_EXP),
    ("AMP_SELECTED", AMP_EXP),
    ("CROSS_MODEL", CROSS_EXP),
]

score_tokens = [
    "score",
    "anomaly_score",
    "if_score",
    "decision_function",
]

split_tokens = [
    "split",
    "partition",
]

row_id_tokens = [
    "window_id",
    "source_row_id",
    "start_time",
    "end_time",
    "series_key",
    "channel_code",
    "meter",
]

candidate_rows = []

# ------------------------------------------------------------
# Inspect CSV files only
# ------------------------------------------------------------
for root_name, root in search_roots:

    if not root.exists():
        continue

    for p in root.rglob("*.csv"):

        try:
            sample = pd.read_csv(p, nrows=10)
        except Exception:
            continue

        cols = list(sample.columns)
        cols_lower = [str(c).lower() for c in cols]

        # Detect likely score columns
        detected_score_cols = [
            c for c in cols
            if any(
                token in str(c).lower()
                for token in score_tokens
            )
        ]

        detected_split_cols = [
            c for c in cols
            if any(
                token == str(c).lower()
                for token in split_tokens
            )
        ]

        detected_row_id_cols = [
            c for c in cols
            if any(
                token == str(c).lower()
                for token in row_id_tokens
            )
        ]

        # Ignore files with no score-like fields
        if not detected_score_cols:
            continue

        # ----------------------------------------------------
        # Estimate actual CSV data-row count
        # ----------------------------------------------------
        try:
            with open(p, "rb") as f:
                n_lines = sum(1 for _ in f)

            n_rows = max(n_lines - 1, 0)

        except Exception:
            n_rows = np.nan

        # ----------------------------------------------------
        # Inspect split values if possible
        # ----------------------------------------------------
        split_values = ""

        if detected_split_cols:

            split_col = detected_split_cols[0]

            try:
                split_df = pd.read_csv(
                    p,
                    usecols=[split_col]
                )

                vals = (
                    split_df[split_col]
                    .dropna()
                    .astype(str)
                    .unique()
                    .tolist()
                )

                split_values = ", ".join(
                    sorted(vals)
                )

            except Exception as e:
                split_values = f"ERROR: {e}"

        # ----------------------------------------------------
        # Dataset hints
        # ----------------------------------------------------
        text = (
            p.name.lower()
            + " "
            + str(p.parent).lower()
        )

        if "ukdale" in text or "uk-dale" in text:
            dataset_hint = "UK-DALE"

        elif "ampds2" in text or "ampds" in text:
            dataset_hint = "AMPds2"

        else:
            dataset_hint = "UNKNOWN"

        # ----------------------------------------------------
        # Isolation Forest filename/schema hint
        # ----------------------------------------------------
        if_hint = (
            "isolation" in text
            or "_if_" in text
            or p.name.lower().startswith("if_")
            or any(
                "if_score" in str(c).lower()
                or "isolation" in str(c).lower()
                for c in cols
            )
        )

        candidate_rows.append({
            "priority_root": root_name,
            "dataset_hint": dataset_hint,
            "file": p.name,
            "relative_path": str(p.relative_to(BASE)),
            "n_rows": n_rows,
            "size_mb": p.stat().st_size / (1024 ** 2),
            "if_hint": if_hint,
            "score_columns": ", ".join(
                map(str, detected_score_cols)
            ),
            "split_columns": ", ".join(
                map(str, detected_split_cols)
            ),
            "split_values": split_values,
            "row_id_columns": ", ".join(
                map(str, detected_row_id_cols)
            ),
            "all_columns": ", ".join(
                map(str, cols)
            ),
        })


candidates = pd.DataFrame(candidate_rows)

assert len(candidates) > 0, \
    "No score-bearing CSV files were found."

# ------------------------------------------------------------
# Prioritize likely authoritative row-level artifacts
# ------------------------------------------------------------
root_priority = {
    "BOUNDARY_CORRECTION": 0,
    "UK_SELECTED": 1,
    "AMP_SELECTED": 1,
    "CROSS_MODEL": 2,
}

candidates["root_rank"] = (
    candidates["priority_root"]
    .map(root_priority)
    .fillna(99)
)

# Prefer:
# 1) corrected
# 2) Isolation Forest hint
# 3) split information
# 4) row identifiers
# 5) larger row-level files
candidates["has_split"] = (
    candidates["split_columns"].str.len() > 0
)

candidates["has_row_id"] = (
    candidates["row_id_columns"].str.len() > 0
)

candidates = candidates.sort_values(
    by=[
        "root_rank",
        "if_hint",
        "has_split",
        "has_row_id",
        "n_rows",
    ],
    ascending=[
        True,
        False,
        False,
        False,
        False,
    ]
).reset_index(drop=True)

# ------------------------------------------------------------
# Show strongest candidates
# ------------------------------------------------------------
print("\nTOP LIKELY ROW-LEVEL SCORE ARTIFACTS:\n")

display(
    candidates[
        [
            "priority_root",
            "dataset_hint",
            "file",
            "n_rows",
            "if_hint",
            "score_columns",
            "split_columns",
            "split_values",
            "row_id_columns",
            "relative_path",
        ]
    ].head(40)
)

# ------------------------------------------------------------
# Specifically show boundary-corrected score files
# ------------------------------------------------------------
corrected = candidates[
    candidates["priority_root"] == "BOUNDARY_CORRECTION"
].copy()

print("\nBOUNDARY-CORRECTION SCORE FILES:\n")

if len(corrected):
    display(
        corrected[
            [
                "dataset_hint",
                "file",
                "n_rows",
                "if_hint",
                "score_columns",
                "split_columns",
                "split_values",
                "row_id_columns",
                "relative_path",
                "all_columns",
            ]
        ]
    )
else:
    print(
        "No score-bearing CSV files were found directly "
        "inside the boundary-correction experiment."
    )

# ------------------------------------------------------------
# Save narrowed inventory
# ------------------------------------------------------------
narrowed_path = (
    THRESHOLD_OUT
    / "residential_row_level_score_candidates.csv"
)

candidates.to_csv(
    narrowed_path,
    index=False
)

print("\nSaved narrowed inventory to:")
print(narrowed_path)

print(
    "\nSTOP HERE. Do not calculate new thresholds yet."
)

In [ ]:
# ============================================================
# FINAL THESIS ROBUSTNESS ANALYSES
# Cell 7 — Verify authoritative corrected 95th-percentile
#          residential Isolation Forest baseline
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

print("=" * 110)
print("CORRECTED RESIDENTIAL ISOLATION FOREST — 95TH-PERCENTILE BASELINE AUDIT")
print("=" * 110)

# ------------------------------------------------------------
# 1. Locate exact authoritative corrected files
# ------------------------------------------------------------
uk_matches = list(
    CORRECTION_EXP.rglob(
        "ukdale_if_scored_boundary_corrected.csv"
    )
)

amp_matches = list(
    CORRECTION_EXP.rglob(
        "ampds2_if_scored_boundary_corrected.csv"
    )
)

assert len(uk_matches) == 1, (
    f"Expected exactly one corrected UK-DALE scored file; "
    f"found {len(uk_matches)}"
)

assert len(amp_matches) == 1, (
    f"Expected exactly one corrected AMPds2 scored file; "
    f"found {len(amp_matches)}"
)

UK_CORRECTED = uk_matches[0]
AMP_CORRECTED = amp_matches[0]

print("\nUK-DALE corrected scored file:")
print(UK_CORRECTED)

print("\nAMPds2 corrected scored file:")
print(AMP_CORRECTED)


# ------------------------------------------------------------
# 2. Load full corrected scored datasets
# ------------------------------------------------------------
uk = pd.read_csv(UK_CORRECTED)
amp = pd.read_csv(AMP_CORRECTED)

datasets = {
    "UK-DALE": uk,
    "AMPds2": amp,
}

print("\nLoaded shapes:")
print("UK-DALE:", uk.shape)
print("AMPds2: ", amp.shape)

assert len(uk) == 363909, (
    f"Unexpected corrected UK-DALE row count: {len(uk)}"
)

assert len(amp) == 735777, (
    f"Unexpected corrected AMPds2 row count: {len(amp)}"
)


# ------------------------------------------------------------
# 3. Inspect required fields
# ------------------------------------------------------------
required = [
    "split",
    "anomaly_score",
]

for dataset, df in datasets.items():

    missing = [
        c for c in required
        if c not in df.columns
    ]

    assert not missing, (
        f"{dataset}: missing required fields {missing}"
    )

    print(f"\n{dataset} columns:")
    print(list(df.columns))

    print(f"\n{dataset} split counts:")
    print(df["split"].value_counts(dropna=False).sort_index())

    print(
        f"\n{dataset} anomaly_score missing values:",
        df["anomaly_score"].isna().sum()
    )

    assert df["anomaly_score"].notna().all(), (
        f"{dataset}: anomaly_score contains missing values."
    )

    assert set(
        df["split"].dropna().astype(str).unique()
    ) == {"train", "val", "test"}, (
        f"{dataset}: unexpected split values."
    )


# ------------------------------------------------------------
# 4. Reconstruct 95th-percentile validation thresholds
# ------------------------------------------------------------
baseline_rows = []

for dataset, df in datasets.items():

    val_scores = (
        df.loc[df["split"] == "val", "anomaly_score"]
        .astype(float)
        .to_numpy()
    )

    test_scores = (
        df.loc[df["split"] == "test", "anomaly_score"]
        .astype(float)
        .to_numpy()
    )

    q95 = float(
        np.percentile(
            val_scores,
            95
        )
    )

    val_flags = val_scores > q95
    test_flags = test_scores > q95

    baseline_rows.append({
        "dataset": dataset,

        "validation_windows": len(val_scores),
        "test_windows": len(test_scores),

        "recomputed_q95_threshold": q95,

        "validation_flagged_windows": int(
            val_flags.sum()
        ),
        "validation_flagged_ratio_percent": (
            100 * val_flags.mean()
        ),

        "test_flagged_windows": int(
            test_flags.sum()
        ),
        "test_flagged_ratio_percent": (
            100 * test_flags.mean()
        ),

        "validation_score_min": float(
            np.min(val_scores)
        ),
        "validation_score_median": float(
            np.median(val_scores)
        ),
        "validation_score_max": float(
            np.max(val_scores)
        ),

        "test_score_min": float(
            np.min(test_scores)
        ),
        "test_score_median": float(
            np.median(test_scores)
        ),
        "test_score_max": float(
            np.max(test_scores)
        ),
    })


baseline95 = pd.DataFrame(baseline_rows)

print("\n" + "=" * 110)
print("RECOMPUTED CORRECTED 95TH-PERCENTILE BASELINE")
print("=" * 110)

display(
    baseline95[
        [
            "dataset",
            "validation_windows",
            "test_windows",
            "recomputed_q95_threshold",
            "validation_flagged_windows",
            "validation_flagged_ratio_percent",
            "test_flagged_windows",
            "test_flagged_ratio_percent",
        ]
    ].round(12)
)


# ------------------------------------------------------------
# 5. Locate corrected split-summary files for comparison
# ------------------------------------------------------------
summary_files = {}

for dataset_key, filename in [
    (
        "UK-DALE",
        "ukdale_if_corrected_split_summary.csv"
    ),
    (
        "AMPds2",
        "ampds2_if_corrected_split_summary.csv"
    ),
]:

    matches = list(
        CORRECTION_EXP.rglob(filename)
    )

    assert len(matches) == 1, (
        f"{dataset_key}: expected exactly one "
        f"{filename}, found {len(matches)}"
    )

    summary_files[dataset_key] = matches[0]


print("\nStored corrected split summaries:")

for dataset, p in summary_files.items():

    print(f"\n--- {dataset} ---")
    print(p)

    stored = pd.read_csv(p)
    display(stored)


# ------------------------------------------------------------
# 6. Save baseline audit
# ------------------------------------------------------------
baseline_path = (
    THRESHOLD_OUT
    / "residential_corrected_q95_baseline_audit.csv"
)

baseline95.to_csv(
    baseline_path,
    index=False
)

print("\nSaved baseline audit:")
print(baseline_path)

print(
    "\nSTOP HERE. Confirm that the recomputed 95th-percentile "
    "results reproduce the corrected thesis baseline before "
    "running the 90/95/99 sensitivity analysis."
)

In [ ]:
# ============================================================
# FINAL THESIS ROBUSTNESS ANALYSES
# Cell 8 — Residential IF threshold-percentile sensitivity
#          90th vs 95th vs 99th validation percentiles
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("=" * 110)
print("RESIDENTIAL ISOLATION FOREST THRESHOLD-PERCENTILE SENSITIVITY")
print("=" * 110)

percentiles = [90, 95, 99]

sensitivity_rows = []

# ------------------------------------------------------------
# 1. Calculate thresholds ONLY from validation anomaly scores
#    and apply each threshold unchanged to validation and test.
# ------------------------------------------------------------
for dataset, df in datasets.items():

    val_scores = (
        df.loc[df["split"] == "val", "anomaly_score"]
        .astype(float)
        .to_numpy()
    )

    test_scores = (
        df.loc[df["split"] == "test", "anomaly_score"]
        .astype(float)
        .to_numpy()
    )

    for percentile in percentiles:

        threshold = float(
            np.percentile(
                val_scores,
                percentile
            )
        )

        # Preserve the implemented thesis rule:
        # anomaly_score > validation-derived threshold
        val_flags = val_scores > threshold
        test_flags = test_scores > threshold

        sensitivity_rows.append({
            "dataset": dataset,
            "threshold_percentile": percentile,
            "validation_threshold": threshold,

            "validation_windows": len(val_scores),
            "validation_flagged_windows": int(
                val_flags.sum()
            ),
            "validation_flagged_ratio_percent": float(
                100 * val_flags.mean()
            ),

            "test_windows": len(test_scores),
            "test_flagged_windows": int(
                test_flags.sum()
            ),
            "test_flagged_ratio_percent": float(
                100 * test_flags.mean()
            ),

            "test_minus_validation_flagged_ratio_pp": float(
                100 * (
                    test_flags.mean()
                    - val_flags.mean()
                )
            ),
        })


threshold_sensitivity = pd.DataFrame(
    sensitivity_rows
)

# ------------------------------------------------------------
# 2. Basic methodological validation
# ------------------------------------------------------------

# Expect one row per dataset × percentile
assert len(threshold_sensitivity) == 6

for dataset, dfm in threshold_sensitivity.groupby("dataset"):

    dfm = dfm.sort_values(
        "threshold_percentile"
    )

    # Higher percentile must imply non-decreasing threshold
    assert np.all(
        np.diff(
            dfm["validation_threshold"]
        ) >= 0
    ), f"{dataset}: thresholds are not monotonic."

    # Higher threshold must not increase flagged ratios
    assert np.all(
        np.diff(
            dfm["validation_flagged_ratio_percent"]
        ) <= 1e-12
    ), f"{dataset}: validation flagged ratios are not monotonic."

    assert np.all(
        np.diff(
            dfm["test_flagged_ratio_percent"]
        ) <= 1e-12
    ), f"{dataset}: test flagged ratios are not monotonic."


# ------------------------------------------------------------
# 3. Confirm the 95th-percentile row reproduces Cell 7
# ------------------------------------------------------------
q95_rows = (
    threshold_sensitivity[
        threshold_sensitivity[
            "threshold_percentile"
        ] == 95
    ]
    .set_index("dataset")
    .sort_index()
)

baseline_check = (
    baseline95
    .set_index("dataset")
    .sort_index()
)

assert np.allclose(
    q95_rows["validation_threshold"],
    baseline_check["recomputed_q95_threshold"],
    atol=1e-12
)

assert np.allclose(
    q95_rows["test_flagged_ratio_percent"],
    baseline_check["test_flagged_ratio_percent"],
    atol=1e-12
)

print(
    "\nPASS: the 95th-percentile rows exactly reproduce "
    "the corrected residential baseline."
)


# ------------------------------------------------------------
# 4. Display thesis-facing results
# ------------------------------------------------------------
print("\nThreshold-sensitivity results:\n")

display(
    threshold_sensitivity[
        [
            "dataset",
            "threshold_percentile",
            "validation_threshold",
            "validation_flagged_windows",
            "validation_flagged_ratio_percent",
            "test_flagged_windows",
            "test_flagged_ratio_percent",
            "test_minus_validation_flagged_ratio_pp",
        ]
    ].round(6)
)


# ------------------------------------------------------------
# 5. Compact pivot table
# ------------------------------------------------------------
compact = (
    threshold_sensitivity
    .pivot(
        index="dataset",
        columns="threshold_percentile",
        values="test_flagged_ratio_percent"
    )
    .rename(
        columns={
            90: "test_flagged_ratio_q90_percent",
            95: "test_flagged_ratio_q95_percent",
            99: "test_flagged_ratio_q99_percent",
        }
    )
    .reset_index()
)

print("\nCompact test flagged-ratio sensitivity table:\n")
display(compact.round(6))


# ------------------------------------------------------------
# 6. Save numerical artifacts
# ------------------------------------------------------------
results_path = (
    THRESHOLD_OUT
    / "residential_if_threshold_percentile_sensitivity.csv"
)

compact_path = (
    THRESHOLD_OUT
    / "residential_if_threshold_percentile_sensitivity_compact.csv"
)

threshold_sensitivity.to_csv(
    results_path,
    index=False
)

compact.to_csv(
    compact_path,
    index=False
)


# ------------------------------------------------------------
# 7. Plot — UK-DALE
# ------------------------------------------------------------
uk_plot = threshold_sensitivity[
    threshold_sensitivity["dataset"] == "UK-DALE"
].sort_values(
    "threshold_percentile"
)

plt.figure(figsize=(8, 5))

plt.plot(
    uk_plot["threshold_percentile"],
    uk_plot["validation_flagged_ratio_percent"],
    marker="o",
    label="Validation"
)

plt.plot(
    uk_plot["threshold_percentile"],
    uk_plot["test_flagged_ratio_percent"],
    marker="o",
    label="Test"
)

plt.xlabel("Validation-score threshold percentile")
plt.ylabel("Flagged-window ratio (%)")
plt.title(
    "UK-DALE Isolation Forest Threshold Sensitivity"
)
plt.xticks(percentiles)
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()

uk_plot_path = (
    THRESHOLD_OUT
    / "ukdale_if_threshold_percentile_sensitivity.png"
)

plt.savefig(
    uk_plot_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ------------------------------------------------------------
# 8. Plot — AMPds2
# ------------------------------------------------------------
amp_plot = threshold_sensitivity[
    threshold_sensitivity["dataset"] == "AMPds2"
].sort_values(
    "threshold_percentile"
)

plt.figure(figsize=(8, 5))

plt.plot(
    amp_plot["threshold_percentile"],
    amp_plot["validation_flagged_ratio_percent"],
    marker="o",
    label="Validation"
)

plt.plot(
    amp_plot["threshold_percentile"],
    amp_plot["test_flagged_ratio_percent"],
    marker="o",
    label="Test"
)

plt.xlabel("Validation-score threshold percentile")
plt.ylabel("Flagged-window ratio (%)")
plt.title(
    "AMPds2 Isolation Forest Threshold Sensitivity"
)
plt.xticks(percentiles)
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()

amp_plot_path = (
    THRESHOLD_OUT
    / "ampds2_if_threshold_percentile_sensitivity.png"
)

plt.savefig(
    amp_plot_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ------------------------------------------------------------
# 9. Completion
# ------------------------------------------------------------
print("\nSaved full sensitivity table:")
print(results_path)

print("\nSaved compact table:")
print(compact_path)

print("\nSaved UK-DALE figure:")
print(uk_plot_path)

print("\nSaved AMPds2 figure:")
print(amp_plot_path)

print(
    "\nPASS: residential 90th/95th/99th percentile "
    "sensitivity analysis completed without retraining."
)

In [ ]:
# ============================================================
# FINAL THESIS ROBUSTNESS ANALYSES
# Cell 9 — Final artifact audit and robustness-analysis freeze
# ============================================================

from pathlib import Path
from datetime import datetime

print("=" * 110)
print("FINAL SUPERVISOR ROBUSTNESS ANALYSIS — ARTIFACT FREEZE")
print("=" * 110)

ROBUSTNESS_ROOT = (
    BASE
    / "experiments"
    / "EXP_FINAL_SUPERVISOR_ROBUSTNESS"
)

# ------------------------------------------------------------
# Expected NAB uncertainty artifacts
# ------------------------------------------------------------
expected_nab = [
    OUT / "nab_supervised_final_test_prediction_audit.csv",
    OUT / "nab_supervised_f1_positive_outcome_sensitivity.csv",
    OUT / "nab_supervised_f1_positive_outcome_sensitivity_table.csv",
    OUT / "nab_supervised_f1_positive_outcome_sensitivity_summary.csv",
    OUT / "nab_supervised_f1_sensitivity_thesis_summary.csv",
    OUT / "nab_supervised_f1_positive_outcome_sensitivity.png",
]

# ------------------------------------------------------------
# Expected residential threshold-sensitivity artifacts
# ------------------------------------------------------------
expected_threshold = [
    THRESHOLD_OUT / "residential_score_artifact_inventory.csv",
    THRESHOLD_OUT / "residential_row_level_score_candidates.csv",
    THRESHOLD_OUT / "residential_corrected_q95_baseline_audit.csv",
    THRESHOLD_OUT / "residential_if_threshold_percentile_sensitivity.csv",
    THRESHOLD_OUT / "residential_if_threshold_percentile_sensitivity_compact.csv",
    THRESHOLD_OUT / "ukdale_if_threshold_percentile_sensitivity.png",
    THRESHOLD_OUT / "ampds2_if_threshold_percentile_sensitivity.png",
]

expected_all = expected_nab + expected_threshold

# ------------------------------------------------------------
# Verify artifacts
# ------------------------------------------------------------
missing = []

print("\nArtifact check:\n")

for p in expected_all:

    exists = p.exists()

    print(
        "[OK]     " if exists else "[MISSING]",
        p.relative_to(ROBUSTNESS_ROOT)
    )

    if not exists:
        missing.append(p)

assert not missing, (
    "Some final robustness-analysis artifacts are missing:\n"
    + "\n".join(str(p) for p in missing)
)

# ------------------------------------------------------------
# Create final documentation file
# ------------------------------------------------------------
freeze_time = datetime.now().isoformat(
    timespec="seconds"
)

readme = f"""# Final Supervisor-Requested Robustness Analyses

Freeze timestamp: {freeze_time}

## Status

COMPLETED

No model retraining, feature redesign, hyperparameter search,
or dataset expansion was performed.

The analyses use previously saved final-test predictions and
post-split-boundary-correction Isolation Forest anomaly scores.

---

## 1. NAB Supervised F1 Positive-Outcome Sensitivity

Final supervised Feature Set V2 test population:

- Test windows: 13,548
- Positive windows: 48
- Negative windows: 13,500

Baseline validation-threshold F1:

- Logistic Regression: 0.021739
- MLP: 0.038781
- Random Forest: 0.030060
- XGBoost: 0.000000

Effect of one additional correctly detected positive window:

- Logistic Regression: 0.021739 -> 0.028881
- MLP: 0.038781 -> 0.044199
- Random Forest: 0.030060 -> 0.032032
- XGBoost: 0.000000 -> 0.021978

Interpretation:

The small positive test population makes thresholded F1 values,
particularly for models with very few true positives, sensitive
to the classification outcome of only one or two positive windows.

This analysis is deterministic and is not a bootstrap confidence
interval. Ground-truth labels remain fixed.

---

## 2. Residential Isolation Forest Threshold Sensitivity

Thresholds were recalculated exclusively from the corrected
validation anomaly-score distributions and applied unchanged
to corrected test scores.

### UK-DALE

Validation percentile -> test flagged-window ratio

- 90th -> 9.017587%
- 95th -> 4.879088%
- 99th -> 1.132179%

Corresponding validation flagged ratios were approximately:

- 90th -> 9.999863%
- 95th -> 5.000618%
- 99th -> 1.000399%

Interpretation:

UK-DALE maintains broadly similar validation and test flagged
ratios across the three examined thresholds.

### AMPds2

Validation percentile -> test flagged-window ratio

- 90th -> 14.846378%
- 95th -> 9.894867%
- 99th -> 1.048611%

Corresponding validation flagged ratios were approximately:

- 90th -> 10.000204%
- 95th -> 5.000442%
- 99th -> 1.000360%

Test-minus-validation differences:

- 90th -> +4.846174 percentage points
- 95th -> +4.894425 percentage points
- 99th -> +0.048251 percentage points

Interpretation:

The elevated AMPds2 test-period flagging persists at the
90th and 95th percentile operating thresholds but largely
disappears at the more restrictive 99th percentile.

Therefore, the magnitude of the unlabeled flagged-window
ratio is threshold-dependent and should not be interpreted
as threshold-independent anomaly prevalence.

---

## Authoritative corrected residential source files

UK-DALE:
{UK_CORRECTED}

AMPds2:
{AMP_CORRECTED}

---

## Analysis Constraints

- No retraining was performed.
- No test information was used to calculate thresholds.
- Thresholds were derived exclusively from validation scores.
- UK-DALE and AMPds2 remain unlabeled residential datasets.
- Flagged windows remain model-generated anomaly candidates,
  not confirmed physical anomalies.
"""

README_PATH = (
    ROBUSTNESS_ROOT
    / "README_FINAL_SUPERVISOR_ROBUSTNESS_ANALYSES.md"
)

README_PATH.write_text(
    readme,
    encoding="utf-8"
)

print("\n" + "=" * 110)
print("FINAL STATUS")
print("=" * 110)

print("\nPASS: all expected robustness-analysis artifacts exist.")
print("PASS: NAB supervisor uncertainty request completed.")
print("PASS: residential threshold-sensitivity request completed.")
print("PASS: all three figures are saved.")
print("PASS: no model retraining was performed.")

print("\nFinal documentation saved to:")
print(README_PATH)

print(
    "\nROBUSTNESS ANALYSIS CODE CAN NOW BE FROZEN."
)